<a href="https://colab.research.google.com/github/barlytoes/Plasma-Etch-Recipe-Development/blob/main/src/optimization_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SF₆/O₂ ICP-RIE recipe optimization (Oxford Plasmalab System 100)

**Goal:** anisotropic, smooth Si etching at a decent rate, with the S1813 mask surviving.

**Fixed:** ICP 800 W · Ar 15 sccm · 60 s.
**Varied:** RF bias, chamber pressure, total reactive flow $Q = \text{SF}_6 + \text{O}_2$, with the O₂ fraction $f = \text{O}_2 / Q$ held constant once chosen.

The campaign runs in three phases. Set `PHASE` in the configuration cell, run all cells, and log each run in `data/nanofab_experiments.csv`.

| Phase | What the notebook does | Runs |
|---|---|---|
| `screening` | Prints S1–S4 (only *f* changes), then scores them and recommends *f\** | 3–4 |
| `initial` | Prints a space-filling Latin hypercube over (RF, P, Q) at *f\**, ordered by increasing bias | 5 |
| `optimize` | Fits one random forest per response, scores candidates by Expected Improvement on the desirability *D*, prints the next recipe | 6–8 |

**Objective.** Each response is mapped to a desirability $d_i \in [0,1]$ and combined as a geometric mean $D = (\prod_i d_i)^{1/n}$. A run that fails any one requirement gets $D = 0$. Recipes predicted to leave too little resist are also scored $D = 0$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

## 1. Configuration

Everything you might change lives in this cell.

* `PHASE`: `"screening"`, then `"initial"`, then `"optimize"`.
* `F_FIXED`: leave as `None` during screening. Afterwards, set it to the recommended *f\** (e.g. `0.45`).
* `DATA_DIR`: `None` reads from GitHub. Set it to a folder path (e.g. `"/content/data"`) to read local files instead. In Colab, upload the CSVs and point this at them.
* `GITHUB_REF`: `"main"`, or a commit hash so the results can be reproduced exactly later.

In [ ]:
# ---- Workflow -------------------------------------------------------------
PHASE   = "screening"          # "screening" , "initial" , "optimize"
F_FIXED = None                 # O2 fraction chosen after screening, e.g. 0.45 or 0.50

# ---- Data source ----------------------------------------------------------
DATA_DIR        = None         # e.g. "/content/data" for local files; None = GitHub
GITHUB_REF      = "main"       # branch name or commit hash
GITHUB_RAW      = "https://raw.githubusercontent.com/barlytoes/Plasma-Etch-Recipe-Development/{ref}/data/"
LITERATURE_FILE = "historical_baseline.csv"
NANOFAB_FILE    = "nanofab_experiments.csv"

# ---- Fixed process settings -----------------------------------------------
ICP_POWER_W = 800
AR_SCCM     = 15
TIME_S      = 60

# ---- Search space (optimization phases) ------------------------------------
RF_BOUNDS_W        = (50.0, 150.0)
PRESSURE_BOUNDS    = (10.0, 40.0)     # mTorr
GAS_LIMITS_SCCM    = (15.0, 40.0)     # applies to SF6 and O2 individually

# ---- Screening phase -------------------------------------------------------
SCREEN_RF_W      = 80.0
SCREEN_PRESSURE  = 25.0
SCREEN_Q_SCCM    = 50.0
SCREEN_F_VALUES  = [0.33, 0.45, 0.50]
SCREEN_REPLICATE = 0.45               # optional S4 repeats this f; set None to skip
TIE_MARGIN       = 0.10               # if D(0.50) - D(0.45) <= this, choose 0.45 (further from the cliff)

# ---- Initial design --------------------------------------------------------
N_INITIAL = 5

# ---- Desirability thresholds -----------------------------------------------
#   'zero' = score 0 at or beyond this value, 'one' = score 1 at or beyond this value
THRESHOLDS = {
    "etch_rate_nm_min": {"zero": 250.0, "one": 700.0},
    "rq_nm":            {"zero": 30.0,  "one": 5.0},
    "sidewall_deg":     {"zero": 70.0,  "one": 88.0},
}
USE_ANGLE = False                     # turn on once every run at f* has an SEM sidewall angle

# ---- Mask constraint -------------------------------------------------------
RESIST_INITIAL_NM       = 1200.0      # S1813 thickness
RESIST_MIN_REMAINING_NM = 300.0       # predicted/measured remaining resist below this -> D = 0
MIN_P_MASK              = 0.8         # only recommend recipes at least this likely to keep the mask

# ---- Model settings --------------------------------------------------------
LIT_WEIGHT   = 0.3                    # sample weight of literature rows relative to UH runs
N_TREES      = 300
N_CANDIDATES = 4096                   # Sobol candidates (power of 2)
XI           = 0.01                   # minimum improvement EI must beat
SEED         = 42

FEATURES = ["rf_power_w", "pressure_mtorr", "q_sccm"]

## 2. Load data

The nanofab CSV holds one row per run. Leave a cell blank if it wasn't measured.

* `phase` is `screening`, `initial` or `optimize`.
* `depth_nm` and `time_s` are enough: `etch_rate_nm_min` is filled in automatically if left blank.
* For a run that didn't etch, enter `depth_nm = 0` and leave `rq_nm` blank.
* Enter `dc_bias_v` as shown on the tool. The sign doesn't matter, since the magnitude is used. It is what links your runs to the Gomez data (§5).

In [ ]:
NANOFAB_COLUMNS = ["run_id", "date", "phase", "rf_power_w", "pressure_mtorr", "sf6_sccm", "o2_sccm",
                   "ar_sccm", "icp_power_w", "time_s", "dc_bias_v", "depth_nm", "etch_rate_nm_min",
                   "rq_nm", "ra_nm", "sidewall_deg", "resist_before_nm", "resist_after_nm", "notes"]
NUMERIC_COLUMNS = [c for c in NANOFAB_COLUMNS if c not in ("run_id", "date", "phase", "notes")]


def _path(name):
    if DATA_DIR:
        return f"{DATA_DIR.rstrip('/')}/{name}"
    return GITHUB_RAW.format(ref=GITHUB_REF) + name


def load_csv(name):
    path = _path(name)
    try:
        return pd.read_csv(path)
    except Exception as e:
        raise RuntimeError(f"Could not load {path}\n  -> {e}\n"
                           "Check DATA_DIR / GITHUB_REF and that the file exists.") from None


LITERATURE_COLUMNS = ["source", "bias_value", "bias_unit", "pressure_mtorr", "sf6_sccm", "o2_sccm",
                      "etch_rate_nm_min", "use_in_model"]

literature = load_csv(LITERATURE_FILE)
missing = [c for c in LITERATURE_COLUMNS if c not in literature.columns]
if missing:
    raise ValueError(f"{LITERATURE_FILE} is missing columns {missing}. "
                     "Upload the updated historical_baseline.csv to data/.")
literature["use_in_model"] = literature["use_in_model"].astype(str).str.lower().eq("true")

nanofab = load_csv(NANOFAB_FILE)
missing = [c for c in NANOFAB_COLUMNS if c not in nanofab.columns]
if missing:
    raise ValueError(f"{NANOFAB_FILE} is missing columns: {missing}")
nanofab = nanofab.dropna(how="all").copy()
for c in NUMERIC_COLUMNS:
    nanofab[c] = pd.to_numeric(nanofab[c], errors="coerce")
nanofab["phase"] = nanofab["phase"].astype(str).str.strip().str.lower()

# Derived quantities
nanofab["q_sccm"] = nanofab["sf6_sccm"] + nanofab["o2_sccm"]
nanofab["o2_fraction"] = nanofab["o2_sccm"] / nanofab["q_sccm"]
fill = nanofab["etch_rate_nm_min"].isna() & nanofab["depth_nm"].notna() & nanofab["time_s"].gt(0)
nanofab.loc[fill, "etch_rate_nm_min"] = nanofab.loc[fill, "depth_nm"] / (nanofab.loc[fill, "time_s"] / 60)
nanofab["resist_rate_nm_min"] = ((nanofab["resist_before_nm"] - nanofab["resist_after_nm"])
                                 / (nanofab["time_s"] / 60))

print(f"Literature rows: {len(literature)} ({literature['use_in_model'].sum()} flagged for the rate model)")
print(f"Nanofab runs logged: {len(nanofab)}")
if len(nanofab):
    print(nanofab.groupby("phase").size().to_string())

## 3. Desirability

Every response uses the same linear ramp:

$$d(y) = \operatorname{clip}\!\left(\frac{y - y_{zero}}{y_{one} - y_{zero}},\ 0,\ 1\right)$$

The single formula covers both directions. For Rq, $y_{one} < y_{zero}$, so the denominator is negative and smaller roughness scores higher.

The rate ramp **levels off** at 700 nm/min, so speed beyond "decent" earns nothing and can't be traded against smoothness. The geometric mean makes any $d_i = 0$ zero the whole score. A run that didn't etch, or that exceeded the Rq limit, gets $D = 0$, and so the model learns where the cliff is.

In [ ]:
def ramp(y, zero, one):
    return np.clip((np.asarray(y, dtype=float) - zero) / (one - zero), 0.0, 1.0)


def active_responses():
    keys = ["etch_rate_nm_min", "rq_nm"]
    return keys + (["sidewall_deg"] if USE_ANGLE else [])


def observed_desirability(df):
    # Score logged runs. NaN = not enough measurements yet to score.
    out = pd.DataFrame(index=df.index)
    keys = active_responses()
    for k in keys:
        out["d_" + k] = ramp(df[k], **THRESHOLDS[k])
        out.loc[df[k].isna(), "d_" + k] = np.nan
    no_etch = df["etch_rate_nm_min"].fillna(np.inf) <= 0
    d_cols = ["d_" + k for k in keys]
    D = out[d_cols].prod(axis=1, skipna=False) ** (1.0 / len(keys))
    D[no_etch] = 0.0                                              # no etch -> fails regardless of Rq
    mask_fail = df["resist_after_nm"] < RESIST_MIN_REMAINING_NM
    D[mask_fail.fillna(False)] = 0.0                              # mask nearly gone -> fails
    out["D"] = D
    return out


if len(nanofab):
    nanofab = nanofab.join(observed_desirability(nanofab))

## 4. Phase 1: screening

Three runs at one center recipe (80 W, 25 mTorr, Q = 50 sccm), with only *f* changing. The flows come from $\text{SF}_6 = Q(1-f)$ and $\text{O}_2 = Qf$.

80 W sits in the lower part of the bias range on purpose. The etch-stop cliff moves to lower *f* as bias drops, so if f = 0.50 etches cleanly here, it will also etch at higher bias.

**Decision rule:** pick the *f* with the highest mean *D*. If 0.50 wins by no more than `TIE_MARGIN` over 0.45, take 0.45 instead, since it's further from the cliff.

In [ ]:
def flows(q, f):
    return q * (1 - f), q * f


def check_gas(sf6, o2):
    lo, hi = GAS_LIMITS_SCCM
    return lo - 1e-9 <= sf6 <= hi + 1e-9 and lo - 1e-9 <= o2 <= hi + 1e-9


def csv_template(rows):
    # Print rows ready to paste into nanofab_experiments.csv (fill in the measured columns).
    print(",".join(NANOFAB_COLUMNS))
    for r in rows:
        vals = {c: "" for c in NANOFAB_COLUMNS}
        vals.update(r)
        print(",".join(str(vals[c]) for c in NANOFAB_COLUMNS))
    print()


def recommend_f(screen):
    means = screen.groupby(screen["o2_fraction"].round(2))["D"].mean()
    if means.isna().all() or len(means) == 0:
        return None, means
    best = means.idxmax()
    if np.isclose(best, 0.50) and 0.45 in means.index and means[0.50] - means[0.45] <= TIE_MARGIN:
        best = 0.45
    return float(best), means


if PHASE == "screening":
    plan = []
    for i, f in enumerate(SCREEN_F_VALUES, start=1):
        plan.append(("S%d" % i, f))
    if SCREEN_REPLICATE is not None:
        plan.append(("S%d" % (len(plan) + 1), SCREEN_REPLICATE))

    rows, table = [], []
    for run_id, f in plan:
        sf6, o2 = flows(SCREEN_Q_SCCM, f)
        if not check_gas(sf6, o2):
            print(f"WARNING: {run_id} flows ({sf6:.1f}/{o2:.1f}) violate GAS_LIMITS_SCCM")
        table.append([run_id, f, SCREEN_RF_W, SCREEN_PRESSURE, SCREEN_Q_SCCM, round(sf6, 1), round(o2, 1)])
        rows.append(dict(run_id=run_id, phase="screening", rf_power_w=SCREEN_RF_W,
                         pressure_mtorr=SCREEN_PRESSURE, sf6_sccm=round(sf6, 1), o2_sccm=round(o2, 1),
                         ar_sccm=AR_SCCM, icp_power_w=ICP_POWER_W, time_s=TIME_S))
    print("SCREENING RECIPES  (ICP %d W, Ar %d sccm, %d s)" % (ICP_POWER_W, AR_SCCM, TIME_S))
    print(pd.DataFrame(table, columns=["run", "f", "RF [W]", "P [mTorr]", "Q [sccm]",
                                       "SF6 [sccm]", "O2 [sccm]"]).to_string(index=False))
    print("\nCSV rows to log (fill in the measured columns):")
    csv_template(rows)

    screen = nanofab[nanofab["phase"] == "screening"] if len(nanofab) else nanofab
    print("\n" + "-" * 70)
    if len(screen) == 0:
        print("No screening runs logged yet. Run S1-S4, log them, and re-run this notebook.")
    else:
        cols = ["run_id", "o2_fraction", "etch_rate_nm_min", "rq_nm"] + \
               (["sidewall_deg"] if USE_ANGLE else []) + ["D"]
        print("LOGGED SCREENING RUNS")
        print(screen[cols].round(3).to_string(index=False))
        best, means = recommend_f(screen)
        print("\nMean D by f:\n" + means.round(3).to_string())
        if best is None:
            print("\nNot enough measurements to score yet.")
        elif (means.fillna(0) == 0).all():
            print("\nAll screening runs scored D = 0. Loosen the 'zero' thresholds before continuing.")
        else:
            if (means.dropna() > 0.9).all():
                print("\nNote: every run scored close to 1. Consider tightening the 'one' thresholds.")
            print(f"\nRECOMMENDED f* = {best:.2f}. Set F_FIXED = {best:.2f} and PHASE = 'initial'.")
            if not USE_ANGLE:
                print("\nCAUTION: USE_ANGLE is off, so anisotropy is NOT in D. f is the variable that\n"
                      "controls sidewall passivation, so this recommendation leans toward rate and\n"
                      "smoothness (lower f). Get SEM cross-sections of the screening samples before\n"
                      "fixing f*, or turn USE_ANGLE on once they are measured.")

## 5. Phase 2: initial design

After screening there are only 1–2 runs at *f\**, which is too few for the Rq model to learn anything. So the next 5 runs are spread across the whole (RF, P, Q) box using a Latin hypercube: each variable's range is divided into 5 bins, and every bin gets exactly one run. That spreads the runs evenly along every axis.

The runs are **ordered by increasing RF bias**, so resist consumption is measured at low bias before you reach 150 W.

Q's range depends on *f\**, because both gases have to stay within `GAS_LIMITS_SCCM`:

$$Q_{min} = \max\!\left(\frac{g_{lo}}{1-f},\ \frac{g_{lo}}{f}\right) \qquad Q_{max} = \min\!\left(\frac{g_{hi}}{1-f},\ \frac{g_{hi}}{f}\right)$$

In [ ]:
def q_bounds(f):
    lo, hi = GAS_LIMITS_SCCM
    q_min = max(lo / (1 - f), lo / f)
    q_max = min(hi / (1 - f), hi / f)
    if q_min >= q_max:
        raise ValueError(f"No feasible Q for f = {f}: gas limits {GAS_LIMITS_SCCM} cannot both be met.")
    return q_min, q_max


def search_bounds(f):
    q_lo, q_hi = q_bounds(f)
    return np.array([RF_BOUNDS_W, PRESSURE_BOUNDS, (q_lo, q_hi)])


def require_f():
    if F_FIXED is None:
        raise ValueError("Set F_FIXED to the f* recommended by the screening phase.")


if PHASE == "initial":
    require_f()
    b = search_bounds(F_FIXED)
    lhs = qmc.LatinHypercube(d=3, seed=SEED).random(N_INITIAL)
    X = qmc.scale(lhs, b[:, 0], b[:, 1])
    X = X[np.argsort(X[:, 0])]                               # increasing bias
    rows, table = [], []
    for i, (rf, p, q) in enumerate(X, start=1):
        rf, p, q = round(rf), round(p * 2) / 2, round(q)
        sf6, o2 = flows(q, F_FIXED)
        table.append(["I%d" % i, rf, p, q, round(sf6, 1), round(o2, 1)])
        rows.append(dict(run_id="I%d" % i, phase="initial", rf_power_w=rf, pressure_mtorr=p,
                         sf6_sccm=round(sf6, 1), o2_sccm=round(o2, 1), ar_sccm=AR_SCCM,
                         icp_power_w=ICP_POWER_W, time_s=TIME_S))
    print(f"INITIAL DESIGN at f* = {F_FIXED:.2f}   Q range {b[2,0]:.1f}-{b[2,1]:.1f} sccm")
    print("Run in this order (increasing bias). Check resist consumption after each run.")
    print(pd.DataFrame(table, columns=["run", "RF [W]", "P [mTorr]", "Q [sccm]",
                                       "SF6 [sccm]", "O2 [sccm]"]).to_string(index=False))
    print("\nCSV rows to log:")
    csv_template(rows)
    done = nanofab[(nanofab["phase"] == "initial")] if len(nanofab) else nanofab
    print(f"\nInitial runs logged so far: {len(done)} / {N_INITIAL}. "
          "When all are logged, set PHASE = 'optimize'.")

## 6. Phase 3: optimize

### Surrogate models
There is one random forest per response, trained on your runs at *f\** (from any phase):

| Model | Trained on | Used for |
|---|---|---|
| Etch rate | All runs (no-etch runs as 0) plus Gomez literature rows | $d_{rate}$ |
| Rq | Runs that etched | $d_{Rq}$ |
| Sidewall angle | Runs with SEM data (only if `USE_ANGLE`) | $d_\theta$ |
| Resist consumption rate | Runs with resist before/after | Mask constraint |

### How the literature enters
Gomez (2004) reports **DC bias voltage**, not RF power, and their structures (0.35 µm holes, oxide mask, 5 °C) etch differently from yours. Two corrections make their data usable for the rate model:

1. **Bias mapping.** Fit $|V_{dc}| = a\,P_{RF} + b\,p + c$ on your runs, then invert it to convert each Gomez voltage into an equivalent RF power on your tool.
2. **Rate calibration.** Fit a literature-only forest and predict your runs, then scale the Gomez rates by the least-squares factor $k = \sum y_{UH}\hat y_{lit} / \sum \hat y_{lit}^2$.

The rows then enter with weight `LIT_WEIGHT`, so they shape the trend in bias and pressure without overriding your measurements. If fewer than 3 runs have a DC-bias reading, the literature is skipped.

### Expected Improvement on D
Each tree in a forest is one plausible version of the response surface. For a candidate *x*, tree *t* of every model gives one prediction set, and therefore one sample $D_t(x)$. Tree samples that leave too little resist get $D_t = 0$. Then:

$$\text{EI}(x) = \frac{1}{T}\sum_{t=1}^{T} \max\!\big(D_t(x) - D_{best} - \xi,\ 0\big)$$

A candidate scores high either because the mean prediction is good (exploitation) or because the trees disagree and some predict a big improvement (exploration).

**Mask filter.** EI alone will happily recommend a recipe where only a few optimistic trees keep the mask, because those trees carry the improvement. That costs a sample. So candidates are first filtered to $P(\text{mask OK}) \ge$ `MIN_P_MASK` (the fraction of resist-model trees predicting enough resist remains), and EI ranks only the survivors.

In [ ]:
def fit_forest(X, y, w=None):
    m = RandomForestRegressor(n_estimators=N_TREES, min_samples_leaf=1, random_state=SEED)
    m.fit(X, y, sample_weight=w)
    return m


def tree_predictions(model, X):
    return np.stack([t.predict(X) for t in model.estimators_])        # shape (T, N)


def literature_rows(lit, uh_rate):
    # Map Gomez rows into this tool's RF-power units and rate scale. Returns (X, y, info) or None.
    use = lit[lit["use_in_model"] & lit["bias_unit"].eq("V_dc")].copy()
    cal = uh_rate.dropna(subset=["dc_bias_v"])
    if len(use) == 0:
        return None, "no literature rows flagged"
    if len(cal) < 3:
        return None, f"only {len(cal)} runs have dc_bias_v (need 3)"
    reg = LinearRegression().fit(cal[["rf_power_w", "pressure_mtorr"]], cal["dc_bias_v"].abs())
    a, b_p = reg.coef_
    c = reg.intercept_
    if a <= 0:
        return None, f"DC-bias fit gave a non-physical slope ({a:.3f} V/W)"
    use["rf_power_w"] = (use["bias_value"].abs() - c - b_p * use["pressure_mtorr"]) / a
    use = use[use["rf_power_w"] >= 0]
    if len(use) == 0:
        return None, "all literature biases map below 0 W on this tool"
    use["q_sccm"] = use["sf6_sccm"] + use["o2_sccm"]
    X_lit = use[FEATURES].values
    lit_model = fit_forest(X_lit, use["etch_rate_nm_min"].values)
    yhat = lit_model.predict(uh_rate[FEATURES].values)
    k = float(np.dot(uh_rate["etch_rate_nm_min"], yhat) / np.dot(yhat, yhat))
    if k <= 0:
        return None, f"non-physical calibration factor k = {k:.3f}"
    info = (f"|Vdc| = {a:.3f}*RF {b_p:+.3f}*P {c:+.1f}  ->  {len(use)} Gomez rows mapped to "
            f"RF {use['rf_power_w'].min():.0f}-{use['rf_power_w'].max():.0f} W;  rate scale k = {k:.2f}")
    return (X_lit, k * use["etch_rate_nm_min"].values), info


MODELS = {}

if PHASE == "optimize":
    require_f()
    data = nanofab[np.isclose(nanofab["o2_fraction"], F_FIXED, atol=0.01)].copy()
    rate_df = data.dropna(subset=["etch_rate_nm_min"])
    rq_df = data[(data["etch_rate_nm_min"] > 0) & data["rq_nm"].notna()]
    ang_df = data[(data["etch_rate_nm_min"] > 0) & data["sidewall_deg"].notna()]
    res_df = data.dropna(subset=["resist_rate_nm_min"])
    print(f"Runs at f* = {F_FIXED:.2f}: {len(data)}  (rate {len(rate_df)}, Rq {len(rq_df)}, "
          f"angle {len(ang_df)}, resist {len(res_df)})")
    if len(rate_df) < 4 or len(rq_df) < 3:
        raise ValueError("Need at least 4 runs with etch rate and 3 with Rq at f*. "
                         "Finish the initial phase first.")
    if USE_ANGLE and len(ang_df) < len(rq_df):
        raise ValueError("USE_ANGLE is on but some etched runs have no sidewall_deg. "
                         "Measure them, or keep USE_ANGLE = False.")

    # --- fit models ---
    X_rate, y_rate = rate_df[FEATURES].values, rate_df["etch_rate_nm_min"].values
    w_rate = np.ones(len(y_rate))
    lit, info = literature_rows(literature, rate_df)
    if lit is None:
        print(f"Literature not used: {info}")
    else:
        print(f"Literature: {info}")
        X_rate = np.vstack([X_rate, lit[0]])
        y_rate = np.concatenate([y_rate, lit[1]])
        w_rate = np.concatenate([w_rate, np.full(len(lit[1]), LIT_WEIGHT)])
    MODELS["etch_rate_nm_min"] = (fit_forest(X_rate, y_rate, w_rate), rate_df)
    MODELS["rq_nm"] = (fit_forest(rq_df[FEATURES].values, rq_df["rq_nm"].values), rq_df)
    if USE_ANGLE:
        MODELS["sidewall_deg"] = (fit_forest(ang_df[FEATURES].values, ang_df["sidewall_deg"].values), ang_df)
    use_mask = len(res_df) >= 3
    if use_mask:
        MODELS["resist_rate_nm_min"] = (fit_forest(res_df[FEATURES].values,
                                                   res_df["resist_rate_nm_min"].values), res_df)
    else:
        print(f"Mask constraint inactive: only {len(res_df)} runs have resist before/after (need 3).")

    # --- candidates ---
    b = search_bounds(F_FIXED)
    sob = qmc.Sobol(d=3, scramble=True, seed=SEED).random(N_CANDIDATES)
    C = qmc.scale(sob, b[:, 0], b[:, 1])

    # --- sample D from paired trees ---
    keys = active_responses()
    d_samples = []
    for k in keys:
        P = tree_predictions(MODELS[k][0], C)
        d_samples.append(ramp(P, **THRESHOLDS[k]))
    D_s = np.prod(d_samples, axis=0) ** (1.0 / len(keys))
    if use_mask:
        remaining = RESIST_INITIAL_NM - tree_predictions(MODELS["resist_rate_nm_min"][0], C) * TIME_S / 60
        feasible = remaining >= RESIST_MIN_REMAINING_NM
        D_s = np.where(feasible, D_s, 0.0)
        p_feas = feasible.mean(axis=0)
    else:
        p_feas = np.full(len(C), np.nan)

    D_best = np.nanmax(data["D"].values) if data["D"].notna().any() else 0.0
    ei = np.maximum(D_s - D_best - XI, 0.0).mean(axis=0)

    rate_t = tree_predictions(MODELS["etch_rate_nm_min"][0], C)
    rq_t = tree_predictions(MODELS["rq_nm"][0], C)
    res = pd.DataFrame(C, columns=FEATURES)
    res["sf6_sccm"], res["o2_sccm"] = flows(res["q_sccm"], F_FIXED)
    res["rate_mean"], res["rate_sd"] = rate_t.mean(0), rate_t.std(0)
    res["rq_mean"], res["rq_sd"] = rq_t.mean(0), rq_t.std(0)
    res["P_mask_ok"] = p_feas
    res["D_mean"] = D_s.mean(0)
    res["EI"] = ei

    eligible = np.ones(len(res), bool)
    if use_mask:
        eligible = res["P_mask_ok"].values >= MIN_P_MASK
        if not eligible.any():
            print(f"\nWARNING: no candidate has P_mask_ok >= {MIN_P_MASK}. Ranking all candidates; "
                  "consider a lower RF ceiling or a shorter TIME_S.")
            eligible[:] = True
        else:
            print(f"Candidates passing the mask filter (P_mask_ok >= {MIN_P_MASK}): "
                  f"{eligible.sum()} / {len(res)}")
    res = res[eligible]
    ei = res["EI"].values

    if ei.max() <= 0:
        hint = ("Best run is already near D = 1: tighten the 'one' thresholds to keep improving."
                if D_best >= 0.9 else
                "If D_mean is ~0 everywhere, loosen the 'zero' thresholds.")
        print("\nEI is zero everywhere: no candidate is predicted to beat D_best by XI.\n"
              "Falling back to the highest mean D. " + hint)
        top = res.sort_values("D_mean", ascending=False).head(3)
    else:
        top = res.sort_values("EI", ascending=False).head(3)

    print(f"\nBest observed D at f*: {D_best:.3f}")
    show = top.copy()
    show[["rf_power_w", "pressure_mtorr", "q_sccm", "sf6_sccm", "o2_sccm"]] = \
        show[["rf_power_w", "pressure_mtorr", "q_sccm", "sf6_sccm", "o2_sccm"]].round(1)
    print("\nTOP 3 CANDIDATES")
    print(show.round(3).to_string(index=False))

    nxt = top.iloc[0]
    rf, p, q = round(nxt.rf_power_w), round(nxt.pressure_mtorr * 2) / 2, round(nxt.q_sccm)
    sf6, o2 = flows(q, F_FIXED)
    n_opt = int((nanofab["phase"] == "optimize").sum()) + 1
    print("\n" + "=" * 60)
    print("NEXT RECIPE  (Oxford Plasmalab System 100)")
    print(f"  Fixed:  ICP {ICP_POWER_W} W | Ar {AR_SCCM} sccm | {TIME_S} s")
    print(f"  RF bias  {rf} W")
    print(f"  Pressure {p} mTorr")
    print(f"  SF6      {sf6:.1f} sccm")
    print(f"  O2       {o2:.1f} sccm      (f = {F_FIXED:.2f}, Q = {q} sccm)")
    print(f"  Predicted: rate {nxt.rate_mean:.0f} ± {nxt.rate_sd:.0f} nm/min | "
          f"Rq {nxt.rq_mean:.1f} ± {nxt.rq_sd:.1f} nm | D {nxt.D_mean:.2f}")
    print("=" * 60)
    print("\nCSV row to log:")
    csv_template([dict(run_id="O%d" % n_opt, phase="optimize", rf_power_w=rf, pressure_mtorr=p,
                       sf6_sccm=round(sf6, 1), o2_sccm=round(o2, 1), ar_sccm=AR_SCCM,
                       icp_power_w=ICP_POWER_W, time_s=TIME_S)])

## 7. Diagnostics

* **D history** shows whether the campaign is improving.
* **Feature importances** show which knob drives each response. Treat them as rough with fewer than about 10 runs.
* **Leave-one-out error** (≥ 6 runs, your data only): each run is predicted by a model trained on all the *other* runs. If the error is close to the spread of the data, the model isn't predicting better than the average, and its recommendations should be read as exploration.

In [ ]:
if len(nanofab) and nanofab["D"].notna().any():
    fig, ax = plt.subplots(figsize=(7, 3))
    colors = {"screening": "#888780", "initial": "#378ADD", "optimize": "#D85A30"}
    for ph, g in nanofab.reset_index(drop=True).groupby("phase"):
        ax.scatter(g.index, g["D"], label=ph, color=colors.get(ph, "k"), zorder=3)
    ax.plot(nanofab.reset_index(drop=True)["D"].cummax(), color="#444441", lw=1, label="best so far")
    ax.set_xlabel("run order"); ax.set_ylabel("D"); ax.set_ylim(-0.05, 1.05)
    ax.legend(frameon=False, fontsize=8); ax.set_title("Desirability by run")
    plt.tight_layout(); plt.show()
else:
    print("No scored runs yet.")

if MODELS:
    imp = pd.DataFrame({k: m.feature_importances_ for k, (m, _) in MODELS.items()}, index=FEATURES)
    print("Feature importances (rate model includes literature rows):")
    print(imp.round(2).to_string())

    for k in ["etch_rate_nm_min", "rq_nm"]:
        df = MODELS[k][1]
        if len(df) >= 6:
            pred = cross_val_predict(RandomForestRegressor(n_estimators=200, random_state=SEED),
                                     df[FEATURES].values, df[k].values, cv=LeaveOneOut())
            mae = np.abs(pred - df[k].values).mean()
            print(f"LOO MAE {k}: {mae:.1f}   (std of data: {df[k].std():.1f})")
        else:
            print(f"LOO for {k}: needs >= 6 runs (have {len(df)})")